# BioFuse Tutorial 1: End-to-End Basic Usage

This tutorial demonstrates the complete workflow of using BioFuse:
1. Loading a biomedical dataset (PathMNIST)
2. Extracting embeddings from foundation models
3. Training a classifier
4. Evaluating performance

## Prerequisites

```bash
pip install biofuse
```

## 1. Setup and Imports

In [ ]:
import numpy as np
import torch
from biofuse import BioFuse, load_medmnist, get_classifier
from biofuse import set_seed, compute_metrics

# Set random seed for reproducibility
set_seed(42)

print("Setup complete!")

## 2. Load Dataset

We'll use PathMNIST, a dataset of colorectal cancer histology images with 9 tissue types.

In [ ]:
# Load PathMNIST dataset
train_data, num_classes = load_medmnist('pathmnist', split='train', download=True)
val_data, _ = load_medmnist('pathmnist', split='val', download=True)
test_data, _ = load_medmnist('pathmnist', split='test', download=True)

print(f"Dataset: PathMNIST")
print(f"Number of classes: {num_classes}")
print(f"Train samples: {len(train_data)}")
print(f"Val samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")

## 3. Visualize Sample Images

In [ ]:
import matplotlib.pyplot as plt

# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

for i in range(10):
    img, label = train_data[i]
    # Convert tensor to numpy for display
    img_np = img.permute(1, 2, 0).numpy()
    axes[i].imshow(img_np)
    axes[i].set_title(f"Class {label}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 4. Initialize BioFuse and Extract Embeddings

We'll use two foundation models: BioMedCLIP and CONCH.

**Note**: Embedding extraction can take time. BioFuse automatically caches embeddings for future use.

In [ ]:
# Initialize BioFuse with two models
biofuse = BioFuse(
    models=['BioMedCLIP', 'CONCH'],
    fusion_method='concat',  # Concatenate embeddings
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

print(f"Using device: {biofuse.device}")
print(f"Models: {biofuse.models}")

In [ ]:
# Extract embeddings for training set
print("Extracting training embeddings...")
train_emb, train_labels, _, _, _ = biofuse.generate_embeddings(
    train_data=None,
    dataset_type='medmnist',
    dataset_name='pathmnist',
    split='train'
)

print(f"Train embeddings shape: {train_emb.shape}")
print(f"Train labels shape: {train_labels.shape}")

In [ ]:
# Extract embeddings for validation set
print("Extracting validation embeddings...")
val_emb, val_labels, _, _, _ = biofuse.generate_embeddings(
    train_data=None,
    dataset_type='medmnist',
    dataset_name='pathmnist',
    split='val'
)

print(f"Val embeddings shape: {val_emb.shape}")

In [ ]:
# Extract embeddings for test set
print("Extracting test embeddings...")
test_emb, test_labels, _, _, _ = biofuse.generate_embeddings(
    train_data=None,
    dataset_type='medmnist',
    dataset_name='pathmnist',
    split='test'
)

print(f"Test embeddings shape: {test_emb.shape}")

## 5. Train Classifier

Let's start with a simple logistic regression classifier.

In [ ]:
# Initialize logistic regression classifier
classifier = get_classifier('logistic', num_classes=num_classes)

print(f"Classifier: {type(classifier).__name__}")

In [ ]:
# Train classifier
print("Training classifier...")
classifier.fit(train_emb, train_labels, X_val=val_emb, y_val=val_labels)

print("Training complete!")

## 6. Evaluate Performance

In [ ]:
# Make predictions on test set
test_pred = classifier.predict(test_emb)
test_proba = classifier.predict_proba(test_emb)

print(f"Predictions shape: {test_pred.shape}")
print(f"Probabilities shape: {test_proba.shape}")

In [ ]:
# Compute metrics
metrics = compute_metrics(
    y_true=test_labels,
    y_pred=test_pred,
    y_proba=test_proba,
    num_classes=num_classes,
    task='multi-class'
)

# Display results
print("\n" + "="*50)
print("TEST SET RESULTS")
print("="*50)
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
print(f"AUC (macro): {metrics['auc_macro']:.4f}")
print(f"AUC (weighted): {metrics['auc_weighted']:.4f}")
print("="*50)

## 7. Visualize Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Compute confusion matrix
cm = confusion_matrix(test_labels, test_pred)

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - PathMNIST')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 8. Try Different Classifiers

BioFuse supports multiple classifiers. Let's compare a few.

In [ ]:
# Test multiple classifiers
classifier_types = ['logistic', 'xgboost', 'catboost']
results = {}

for clf_type in classifier_types:
    print(f"\nTraining {clf_type}...")
    
    # Initialize classifier
    clf = get_classifier(clf_type, num_classes=num_classes)
    
    # Train
    clf.fit(train_emb, train_labels, X_val=val_emb, y_val=val_labels)
    
    # Predict
    pred = clf.predict(test_emb)
    proba = clf.predict_proba(test_emb)
    
    # Evaluate
    metrics = compute_metrics(
        y_true=test_labels,
        y_pred=pred,
        y_proba=proba,
        num_classes=num_classes,
        task='multi-class'
    )
    
    results[clf_type] = metrics['accuracy']
    print(f"  Accuracy: {metrics['accuracy']:.4f}")

In [ ]:
# Plot comparison
plt.figure(figsize=(10, 6))
plt.bar(results.keys(), results.values())
plt.ylabel('Accuracy')
plt.xlabel('Classifier Type')
plt.title('Classifier Comparison on PathMNIST')
plt.ylim([0, 1])
plt.grid(axis='y', alpha=0.3)
for i, (name, acc) in enumerate(results.items()):
    plt.text(i, acc + 0.02, f'{acc:.4f}', ha='center')
plt.show()

## 9. Save Model (Optional)

In [ ]:
# Save the best classifier
import joblib
from pathlib import Path

# Create output directory
output_dir = Path('outputs/pathmnist_tutorial')
output_dir.mkdir(parents=True, exist_ok=True)

# Save classifier
best_clf_type = max(results, key=results.get)
best_clf = get_classifier(best_clf_type, num_classes=num_classes)
best_clf.fit(train_emb, train_labels, X_val=val_emb, y_val=val_labels)

model_path = output_dir / f'pathmnist_{best_clf_type}.pkl'
joblib.dump(best_clf, model_path)

print(f"Model saved to: {model_path}")
print(f"Best classifier: {best_clf_type} (Accuracy: {results[best_clf_type]:.4f})")

## Summary

In this tutorial, you learned how to:

1. ✅ Load a biomedical dataset (PathMNIST)
2. ✅ Extract embeddings using foundation models (BioMedCLIP + CONCH)
3. ✅ Train classifiers (Logistic Regression, XGBoost, CatBoost)
4. ✅ Evaluate model performance
5. ✅ Compare different classifiers
6. ✅ Save trained models

## Next Steps

- **Tutorial 2**: Learn how to add custom datasets
- **Tutorial 3**: Learn how to integrate new foundation models
- **Tutorial 4**: Learn how to implement custom classifiers

## Resources

- [BioFuse Documentation](https://github.com/mnhcorp/biofuse)
- [MedMNIST Dataset](https://medmnist.com/)
- [Configuration Examples](../configs/)